# Testing for structure in a flux-limited sample

A reusable version of the null test from Yaptangco et al., *TESS Reveals the Scaling
Between Planet Size and Stellar Mass at the Extremes of the Main Sequence*. It answers
one question:

> My sample looks like it has a break or a bend in it. How often would a survey with
> no such feature produce something that looks this structured?

It is built for samples where **the chance of detecting an object depends on the
quantity you are measuring**. That is the awkward case. If big planets are easier to
find than small ones, then any test that reshuffles radii without re-asking "would this
object have been detected?" is quietly comparing your data against a universe that could
not exist.

## What the method does

1. Fit a ladder of models through the **weighted median** of your data, with no binning.
2. Measure how much better the structured models fit than the unstructured ones, using
   `D = 2N ln(Q_structured / Q_unstructured)`.
3. Decide whether that D is surprising by **simulating surveys** where the truth is
   known and unstructured, running them through the identical fitting code, and counting
   how often they reach your observed D.

Step 3 is the part that matters. There is no chi-squared table for this statistic, and
using one is badly wrong — in the paper's case a nominal 5 per cent test rejected 21.6
per cent of simulated nulls. The null distribution has to be built by simulation.

## What you need

| Input | Meaning |
|---|---|
| `x` | the independent variable (stellar temperature, mass, metallicity, …) |
| `y` | the quantity whose trend you are testing (planet radius, …) |
| `completeness_weight(slots, y)` | the weight the survey gives object `slots[k]` if its value were `y[k]`, and `inf` where it could not be found |
| `raw_weights` | the weight each observed object actually carries, as your pipeline computed it |
| `models` | the shapes you want to compare, split into simple and structured |

Adding a shape is one entry in the `models` dict and no other edits — it is fitted, it
joins its family in the `D` statistic, and it becomes a generating truth in the
calibration on its own. The last cell has worked examples.

`raw_weights` is the same quantity as `completeness_weight` evaluated at each object's own
value, so supplying both looks redundant. It is not. Write `raw_weights` the way your
completeness pipeline writes it, because the two routes agree to algebra and not to the
last bit, and that is enough to matter — see *Reproducing a published run* at the end.

### Why `completeness_weight` is indexed by slot, not by (x, y)

Real selection functions depend on more than the two variables you are plotting —
orbital period, apparent magnitude, how long the star was observed. Those are properties
of the *object*, not of the trend, so the interface takes an **index into your observed
sample** and closes over everything else.

This is not a hypothetical. In the TESS mini-Neptune sample this notebook came from, the
completeness weight is keyed by (temperature slice, radius cell, **period cell**), and
two planets sharing a temperature slice *and* a radius cell differ in weight by up to
**55 times** because their orbital periods differ. A `completeness_weight(x, y)`
interface cannot see that and would hand those planets the same correction.

Every step below prints its numbers and draws a figure, so you can watch what happens to
the data rather than taking the final p-value on trust. The notebook runs end to end on
synthetic data — execute it now, then replace the one marked block.

In [ ]:
#### YOUR DATA GOES HERE ####
# Everything below this cell is generic. Replace the block marked REPLACE THIS with your
# own sample, completeness weights and detection function, and the rest follows.

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

rng = np.random.default_rng(11)

# How many simulated surveys per generating truth. Set this to 200 for a quick pass; the
# numbers quoted in a paper want 1000, and that is what reproducing the published run
# needs, because all four truths share one generator and a shorter run leaves the later
# truths starting from a different point in the stream.
n_simulations = 1000


# ---------------------------------------------------------------- REPLACE THIS --------
# A synthetic survey standing in for a real one. The truth is flat above 4300 K and
# rises below it, so there IS structure here. The sample is deliberately small enough
# that the structure is only marginally detectable -- that is the regime where
# calibrating the null matters, and where a chi-squared table would mislead you.

def demo_detection(x_values, y_values):
    # Two effects every transit survey has. Larger planets are easier to detect, and
    # planets around hotter (so larger, more distant) stars transit less often.
    size_term = np.clip((y_values - 1.5) / 2.5, 0.0, 1.0) ** 2
    star_term = np.clip(1.0 - 0.6 * (x_values - 3000) / 4000, 0.15, 1.0)
    inside_window = (y_values > 1.8) & (y_values < 4.0)
    return np.where(inside_window, np.maximum(size_term * star_term, 1e-3), 0.0)


def demo_truth(x_values):
    # Flat at 2.45 above the break, sloping down below it
    break_point, level, gradient = 4300.0, 2.45, 4.2e-4
    return np.where(x_values < break_point,
                    level + gradient * (x_values - break_point), level)


# Draw an intrinsic population, then keep each object with its detection probability, so
# the demo sample is selection-filtered in exactly the way the method assumes
n_intrinsic = 6500
x_intrinsic = rng.uniform(3100, 6900, n_intrinsic)
y_intrinsic = demo_truth(x_intrinsic) + rng.normal(0, 0.45, n_intrinsic)
detected = rng.random(n_intrinsic) < demo_detection(x_intrinsic, y_intrinsic)

x = x_intrinsic[detected]
y = y_intrinsic[detected]


def completeness_weight(slots, y_values):
    # The weight the survey would give the object in slot slots[k] if its value were
    # y_values[k], and np.inf where it could not have been found at all.
    #
    # Indexed by slot so that per-object covariates come along for free: this one only
    # needs x, but a real survey's grid is usually keyed by period or magnitude too, and
    # those cannot be recovered from (x, y). Replace the body with your own lookup.
    slots = np.asarray(slots)
    chance = demo_detection(x[slots], np.asarray(y_values))
    return np.where(chance > 0, 1.0 / np.where(chance > 0, chance, 1.0), np.inf)


# The weight each observed object actually carries. This is the same quantity as
# completeness_weight at the object's own value, but write it the way your completeness
# pipeline writes it rather than by calling the function above. The two agree to algebra
# and not to the last bit, and the fitted breakpoint is sensitive at that level: a
# difference of 1e-16 in a hundred of the weights was enough to move it by 14 K. Keeping
# this line identical to the pipeline is what makes the run reproducible.
raw_weights = 1.0 / demo_detection(x, y)
# -------------------------------------------------------------- END REPLACE THIS ------

x_min, x_max = x.min(), x.max()
n_objects = len(x)
all_slots = np.arange(n_objects)


def detectability(slots, y_values):
    # The chance of detection is the reciprocal of the weight, which is what the forward
    # model needs. Infinite weight means no chance, and 1/inf is already 0.
    return 1.0 / completeness_weight(slots, y_values)


print(f"{n_objects} objects, x from {x_min:.0f} to {x_max:.0f}")
print(f"  y median {np.median(y):.3f}, 16-84 per cent "
      f"{np.percentile(y, 16):.2f} to {np.percentile(y, 84):.2f}")

## Step 1 — completeness weights

An object detected despite being hard to detect stands for many others that were missed,
so it carries more weight. The weight is the reciprocal of its detection probability:

$$w_i = \frac{1}{d(i,\, y_i)}$$

Two practical points, both of which bit during the paper.

**Normalise the weights to mean one.** Only their relative size matters to the fit, and
normalising keeps the effective sample size equal to the number of objects you actually
observed. Leaving the raw corrections in would set $N$ to the sum of the corrections —
tens of thousands — and treat every object the survey *might* have seen as an
independent measurement. Since $D$ scales linearly with $N$, that inflates the statistic
by whatever factor your corrections happen to be.

**Check the effective sample size per region.** Kish's formula
$n_{\rm eff} = (\sum w)^2 / \sum w^2$ tells you how many independent measurements a
weighted subsample is really worth. A handful of heavily weighted objects can dominate a
region, and knowing that up front tells you which parts of your fit are fragile.

![](step1_selection_and_weights.png)

In [ ]:
#### WEIGHTS AND WHAT THEY ARE WORTH ####

# An object the pipeline could not place gets no correction rather than an infinite one,
# which is what a real survey does with a planet that falls off the edge of its grid
raw_weights = np.where(np.isfinite(raw_weights), raw_weights, 1.0)

# Scaled so the weights average to one, which keeps the effective sample size equal to
# the number of objects observed rather than to the sum of the corrections. The divisor
# is kept because every simulated survey below must be scaled by the same constant, not
# by its own mean, or the losses are not comparable between surveys.
raw_mean = raw_weights.mean()
weights = raw_weights / raw_mean

print(f"Weights scaled to mean 1, range {weights.min():.3f} to {weights.max():.1f}")
print(f"  the most weighted object counts for "
      f"{weights.max() / weights.min():.0f} times the least\n")


def effective_size(w):
    # Kish's effective sample size: how many unweighted objects this is worth
    return w.sum() ** 2 / np.sum(w ** 2)


print(f"Effective sample size over the whole sample: "
      f"{effective_size(weights):.1f} from {n_objects} objects\n")

# Effective size region by region, since a few heavy objects can carry a whole stretch
region_edges = np.linspace(x_min, x_max, 6)
region_centers, region_counts, region_effective = [], [], []
for i in range(len(region_edges) - 1):
    in_region = (x >= region_edges[i]) & (x < region_edges[i + 1])
    if in_region.sum() == 0:
        continue
    region_centers.append((region_edges[i] + region_edges[i + 1]) / 2)
    region_counts.append(int(in_region.sum()))
    region_effective.append(effective_size(weights[in_region]))

print(f"{'region':<20}{'N':>6}{'n_eff':>9}{'median y':>11}")
print("-" * 46)
for i in range(len(region_centers)):
    in_region = (x >= region_edges[i]) & (x < region_edges[i + 1])
    print(f"{f'{region_edges[i]:.0f}-{region_edges[i+1]:.0f}':<20}"
          f"{region_counts[i]:>6}{region_effective[i]:>9.1f}"
          f"{np.median(y[in_region]):>11.3f}")

In [ ]:
#### WHAT THE SELECTION IS DOING TO THE SAMPLE ####
# Three views of the same thing: the selection function itself, where the heavily
# weighted objects sit, and what the weighting costs in effective sample size.

# Create the plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

ax = axes[0]
ax.grid(True, which='major', linewidth=2.2, alpha=0.5, color='0.7')
ax.set_axisbelow(True)
# The selection function at three positions in x, using real slots so any per-object
# covariate is included
y_line = np.linspace(y.min(), y.max(), 300)
for fraction, color, style in ((0.1, 'navy', '-'), (0.5, 'seagreen', '--'),
                              (0.9, 'crimson', (0, (6, 3)))):
    slot = int(np.argsort(x)[int(fraction * (n_objects - 1))])
    probability = detectability(np.full(len(y_line), slot), y_line)
    ax.plot(y_line, probability, color=color, linestyle=style, linewidth=3,
            label=f'object at x = {x[slot]:.0f}')
ax.set_xlabel('y', size=12)
ax.set_ylabel('Probability of detection', size=12)
ax.set_title('The selection function', size=14)
ax.legend(loc='upper left', fontsize=10)

ax = axes[1]
ax.grid(True, which='major', linewidth=2.2, alpha=0.5, color='0.7')
ax.set_axisbelow(True)
# Each object coloured by how much it stands for
scatter = ax.scatter(x, y, c=np.log10(weights), s=26, cmap='viridis', alpha=0.85,
                     zorder=3)
bar = fig.colorbar(scatter, ax=ax)
bar.set_label('log10 weight', size=12)
ax.set_xlabel('x', size=12)
ax.set_ylabel('y', size=12)
ax.set_title('Who stands for whom', size=14)

ax = axes[2]
ax.grid(True, which='major', linewidth=2.2, alpha=0.5, color='0.7')
ax.set_axisbelow(True)
# What the weighting costs: N against n_eff, region by region
width = (region_edges[1] - region_edges[0]) * 0.38
ax.bar(np.array(region_centers) - width / 2, region_counts, width=width,
       color='0.6', label='Objects observed')
ax.bar(np.array(region_centers) + width / 2, region_effective, width=width,
       color='crimson', alpha=0.8, label='Effective sample size')
for i in range(len(region_centers)):
    ax.text(region_centers[i] + width / 2, region_effective[i] + 2,
            f'{100 * region_effective[i] / region_counts[i]:.0f}%',
            ha='center', size=10)
ax.set_xlabel('x', size=12)
ax.set_ylabel('Number of objects', size=12)
ax.set_title('What the weighting costs', size=14)
ax.legend(loc='upper left', fontsize=10)

for ax in axes:
    ax.tick_params(axis='both', which='major', width=3, length=10, labelsize=12)
    for spine in ax.spines.values():
        spine.set_linewidth(2)

fig.suptitle('Step 1: the sample is not a fair draw from the population', size=16)
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

fig.savefig('step1_selection_and_weights.png', dpi=300, bbox_inches='tight')

## Step 2 — fitting through the weighted median, without binning

A weighted median is the value $m$ minimising $\sum_i w_i |y_i - m|$. So fitting a
*curve* through the median is just minimising the same thing with the curve in place of
the constant:

$$Q(\theta) = \sum_i w_i \left| y_i - f(x_i; \theta) \right|$$

This is quantile regression at $\tau = 0.5$ (Koenker & Bassett 1978). It gives the same
estimator as computing a weighted median per bin and fitting those, but with the bins
taken out — so there are no edges to place, and $N$ is the number of objects rather than
the number of bins you chose. In the paper this mattered: the small-sample AICc penalty
charged the four-parameter model 9.3 at eight bins but only 2.5 at thirty-seven, which
is the same size as the effect being measured.

**The models go in one dict**, each carrying its function, parameter count, starting
points, and which family it belongs to. Replace these with your own shapes — the only
requirement is that each is labelled `simple` or `structured`.

The starting points are a *function* of the level and gradient of whatever sample is
being fitted, not fixed numbers. That is deliberate. Every simulated survey below is
fitted by this same code, and a simulated survey has its own median and its own
gradient; starting it from the real sample's values sends the search into a different
basin and moves its $D$ by tens.

$Q$ is piecewise linear and its surface is multi-modal in any breakpoint, so the fit uses
a gradient-free search (Powell) started from several breakpoint positions. The second
panel of the figure below shows why that is not optional.

![](step2_fit_and_loss_profile.png)

In [ ]:
#### THE MODEL LADDER ####
# Each model carries everything about itself: its function, its parameter count, where
# its x-positions sit, where to start the search, which family it belongs to, and its
# colour. Adding a shape means adding one entry here and nothing else -- the fit, the D
# statistic, the forward model and the calibration all read this dict, and a new model
# automatically becomes a generating truth in Step 5.

span = x_max - x_min
level_guess = np.median(y)
cool_half, hot_half = x < np.median(x), x >= np.median(x)
gradient_guess = (np.median(y[hot_half]) - np.median(y[cool_half])) / span


def flat(x_values, c):
    return np.full_like(np.asarray(x_values, dtype=float), c)


def straight_line(x_values, y0, s):
    return y0 + s * (np.asarray(x_values, dtype=float) - x_min)


def one_break(x_values, c, x1, s1):
    # Sloping below x1, flat at c above it
    x_values = np.asarray(x_values, dtype=float)
    return np.where(x_values < x1, c + s1 * (x_values - x1), c)


def two_breaks(x_values, c, x1, x2, s):
    # Sloping, then flat between x1 and x2, then sloping again with the same gradient
    x_values = np.asarray(x_values, dtype=float)
    return np.select(
        [x_values < x1, (x_values >= x1) & (x_values <= x2), x_values > x2],
        [c + s * (x_values - x1), np.full_like(x_values, c), c + s * (x_values - x2)],
    )


# Starting points, written as functions of a summary of whatever sample is being fitted
# rather than as fixed numbers. This matters: a simulated survey has its own level and
# its own gradient, and starting it from the real sample's sends the search into a
# different basin, which moves that survey's D by tens. The summary is built by
# describe_sample below, so a new model can reach anything in it.
def flat_starts(sample):
    return [[sample['level']]]


def straight_line_starts(sample):
    return [[sample['level'], sample['gradient']]]


def one_break_starts(sample):
    starts = []
    for fraction in (0.2, 0.35, 0.5, 0.65, 0.8):
        starts.append([sample['level'], x_min + fraction * span, sample['gradient']])
    return starts


def two_breaks_starts(sample):
    starts = []
    for lower, upper in ((0.2, 0.8), (0.3, 0.7), (0.15, 0.55), (0.45, 0.85), (0.35, 0.6)):
        starts.append([sample['level'], x_min + lower * span, x_min + upper * span,
                       sample['gradient']])
    return starts


models = {
    'flat': {
        'func': flat, 'k': 1, 'family': 'simple',
        'starts': flat_starts, 'breaks': [], 'color': '0.55',
    },
    'straight line': {
        'func': straight_line, 'k': 2, 'family': 'simple',
        'starts': straight_line_starts, 'breaks': [], 'color': '0.35',
    },
    'one break': {
        'func': one_break, 'k': 3, 'family': 'structured',
        'starts': one_break_starts, 'breaks': [1], 'color': 'saddlebrown',
    },
    'two breaks': {
        'func': two_breaks, 'k': 4, 'family': 'structured',
        'starts': two_breaks_starts, 'breaks': [1, 2], 'color': 'darkorange',
    },
}

simple_models = [name for name in models if models[name]['family'] == 'simple']
structured_models = [name for name in models if models[name]['family'] == 'structured']


def refresh_families():
    global simple_models, structured_models
    simple_models = [name for name in models if models[name]['family'] == 'simple']
    structured_models = [name for name in models if models[name]['family'] == 'structured']


def check_loss(func, params, x_values, y_values, w):
    # The tau = 0.5 check function, whose minimiser is the weighted median
    return np.sum(w * 0.5 * np.abs(y_values - func(x_values, *params)))


def describe_sample(x_values, y_values, w):
    # What a model's starting points are allowed to depend on. Recomputed for every
    # sample fitted, which is the whole point of passing it in.
    level = np.median(y_values)
    cool = x_values < np.median(x_values)
    hot = x_values >= np.median(x_values)
    gradient = (np.median(y_values[hot]) - np.median(y_values[cool])) / span
    return {'x': x_values, 'y': y_values, 'weights': w, 'level': level,
            'gradient': gradient, 'x_min': x_min, 'x_max': x_max, 'span': span}


def fit_model(name, x_values, y_values, w):
    # Started from several positions because the loss surface is multi-modal in any
    # parameter that sits on the x axis
    model = models[name]
    sample = describe_sample(x_values, y_values, w)
    best_loss, best_params = np.inf, None
    fallback_loss, fallback_params = np.inf, None
    for guess in model['starts'](sample):
        result = minimize(lambda p: check_loss(model['func'], p, x_values, y_values, w),
                          guess, method='Powell',
                          options={'maxiter': 20000, 'xtol': 1e-4, 'ftol': 1e-6})
        params = np.array(result.x, dtype=float)
        # Anything listed in 'breaks' is a position on the x axis, so it has to land
        # inside the data and, where there is more than one, stay in order
        for position in model['breaks']:
            params[position] = np.clip(params[position], x_min, x_max)
        positions = params[model['breaks']]
        if len(positions) > 1 and np.any(np.diff(positions) < 0):
            # A start that ends with its breakpoints out of order is discarded, not
            # reordered -- the reordered shape is a different fit that was never
            # searched for. But a run where every start comes out crossed must still
            # return something finite or the simulation below silently loses that
            # sample, so the best reordered one is kept as a fallback.
            ordered = params.copy()
            ordered[model['breaks']] = np.sort(positions)
            ordered_loss = check_loss(model['func'], ordered, x_values, y_values, w)
            if ordered_loss < fallback_loss:
                fallback_loss, fallback_params = ordered_loss, ordered
            continue
        loss = check_loss(model['func'], params, x_values, y_values, w)
        if not np.isfinite(loss):
            continue
        if loss < best_loss:
            best_loss, best_params = loss, params
    if best_params is None:
        return fallback_loss, fallback_params
    return best_loss, best_params


def fit_ladder(x_values, y_values, w):
    fitted = {}
    for name in models:
        loss, params = fit_model(name, x_values, y_values, w)
        fitted[name] = {'loss': loss, 'params': params, 'k': models[name]['k']}
    return fitted


def check_model(name):
    # The mistakes people actually make when adding a shape: a starts function that
    # disagrees with k, and a breakpoint that fitted onto the edge of the range
    model = models[name]
    lengths = set()
    for guess in model['starts'](describe_sample(x, y, weights)):
        lengths.add(len(guess))
    fitted_length = len(np.atleast_1d(ladder[name]['params']))
    agrees = lengths == {model['k']} and fitted_length == model['k']
    print(f"  {name:<16} k={model['k']}, starts give {sorted(lengths)}, "
          f"fit returned {fitted_length}  {'ok' if agrees else 'MISMATCH'}")
    for position in model['breaks']:
        value = np.asarray(ladder[name]['params'])[position]
        if np.isclose(value, x_min) or np.isclose(value, x_max):
            print(f"  {name}: breakpoint {position} sat on the edge of the range at "
                  f"{value:.0f} -- this shape has collapsed to a simpler one")


def check_families():
    if not simple_models or not structured_models:
        print("  both families need at least one model, or D has nothing to compare")
        return
    simple_best = min(ladder[name]['loss'] for name in simple_models)
    structured_best = min(ladder[name]['loss'] for name in structured_models)
    if structured_best > simple_best:
        print("  the structured family fits WORSE than the simple one, which it cannot "
              "do if it contains them -- check the family labels")


def add_model(name, func, k, family, starts, breaks, color):
    # Registers a shape from a cell further down, and redoes the two things that depend
    # on the dict: the family lists and the fit to the real sample. Use this rather than
    # editing the dict above if you want to keep the shipped cell untouched -- re-running
    # that cell rebuilds models from scratch and would drop your addition.
    global ladder
    models[name] = {'func': func, 'k': k, 'family': family, 'starts': starts,
                    'breaks': breaks, 'color': color}
    refresh_families()
    ladder = fit_ladder(x, y, weights)
    print(f"Added '{name}'. The ladder is now "
          f"{len(structured_models)} structured and {len(simple_models)} simple.")
    print(f"  check loss {ladder[name]['loss']:.3f}, params "
          f"{np.array2string(np.asarray(ladder[name]['params']), precision=4)}")
    check_model(name)
    check_families()
    return ladder[name]


ladder = fit_ladder(x, y, weights)

print(f"Weighted quantile regression, all {n_objects} objects")
print("=" * 72)
print(f"{'model':<16}{'family':>12}{'k':>4}{'check loss':>14}{'breaks at':>24}")
print("-" * 72)
for name in models:
    positions = np.asarray(ladder[name]['params'])[models[name]['breaks']]
    where = ', '.join(f"{position:.0f}" for position in positions) if len(positions) else '-'
    print(f"{name:<16}{models[name]['family']:>12}{ladder[name]['k']:>4}"
          f"{ladder[name]['loss']:>14.3f}{where:>24}")

best_simple = min(ladder[name]['loss'] for name in simple_models)
best_structured = min(ladder[name]['loss'] for name in structured_models)
print(f"\n  best simple      {best_simple:.3f}")
print(f"  best structured  {best_structured:.3f}")
print(f"  structure buys   {best_simple - best_structured:.3f} in loss")

print("\nLadder checks")
print("-" * 72)
for name in models:
    check_model(name)
check_families()

In [ ]:
#### WHAT THE FIT IS TRACKING, AND WHY SEVERAL STARTS ARE NEEDED ####


def weighted_median(values, w):
    order = np.argsort(values)
    values, w = values[order], w[order]
    cumulative = np.cumsum(w)
    return values[np.searchsorted(cumulative, 0.5 * w.sum())]


# A rolling weighted median, to show the fit really is chasing the median and not a mean
window = (x_max - x_min) / 8
rolling_x, rolling_y = [], []
for center in np.linspace(x_min + window / 2, x_max - window / 2, 60):
    in_window = np.abs(x - center) < window / 2
    if in_window.sum() < 10:
        continue
    rolling_x.append(center)
    rolling_y.append(weighted_median(y[in_window], weights[in_window]))

# The loss as a function of where a breakpoint is forced to sit. A single start would
# land in whichever basin it happened to begin in.
#
# Which model gets profiled is worked out from the ladder rather than named, so this
# panel keeps working when models are added: the structured model with the fewest
# breakpoints, and its first one. The reference line is the best simple fit.
with_breaks = [name for name in structured_models if len(models[name]['breaks']) > 0]
profiled = min(with_breaks, key=lambda name: len(models[name]['breaks']))
break_index = models[profiled]['breaks'][0]
reference = min(simple_models, key=lambda name: ladder[name]['loss'])
sample = describe_sample(x, y, weights)

forced_positions = np.linspace(x_min + 0.02 * span, x_max - 0.02 * span, 30)
forced_loss = []
for position in forced_positions:
    # The profiled model with its first breakpoint held at this position, leaving the
    # other parameters free
    def pinned(x_values, *free, break_at=position):
        full = list(free)
        full.insert(break_index, break_at)
        return models[profiled]['func'](x_values, *full)
    best = np.inf
    for guess in models[profiled]['starts'](sample):
        free_guess = []
        for i in range(len(guess)):
            if i != break_index:
                free_guess.append(guess[i])
        out = minimize(lambda p: check_loss(pinned, p, x, y, weights),
                       free_guess, method='Powell',
                       options={'maxiter': 20000, 'xtol': 1e-4, 'ftol': 1e-6})
        value = check_loss(pinned, out.x, x, y, weights)
        if np.isfinite(value):
            best = min(best, value)
    forced_loss.append(best)
forced_loss = np.array(forced_loss)

print(f"Loss profile for '{profiled}', breakpoint {break_index} forced across "
      f"{len(forced_positions)} positions")
print(f"  best forced position  {forced_positions[np.argmin(forced_loss)]:.0f}, "
      f"loss {forced_loss.min():.3f}")
print(f"  the free fit found    {ladder[profiled]['params'][break_index]:.0f}, "
      f"loss {ladder[profiled]['loss']:.3f}")
print(f"  worst forced position {forced_positions[np.argmax(forced_loss)]:.0f}, "
      f"loss {forced_loss.max():.3f}")
print(f"  spread across the range: {forced_loss.max() - forced_loss.min():.3f} in loss")

# Create the plot
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
ax.grid(True, which='major', linewidth=2.2, alpha=0.5, color='0.7')
ax.set_axisbelow(True)
# Individual objects as grey transparent dots
ax.scatter(x, y, s=13, color='0.6', alpha=0.3, zorder=2, label=f'Objects (N={n_objects})')
ax.plot(rolling_x, rolling_y, color='black', linewidth=4, zorder=6,
        label='Rolling weighted median')
x_line = np.linspace(x_min, x_max, 600)
for name in models:
    ax.plot(x_line, models[name]['func'](x_line, *ladder[name]['params']),
            color=models[name]['color'], linewidth=2.8, zorder=4, label=name)
ax.set_xlabel('x', size=12)
ax.set_ylabel('y', size=12)
ax.set_title('The ladder, against what it is chasing', size=14)
ax.legend(loc='upper left', fontsize=10)

ax = axes[1]
ax.grid(True, which='major', linewidth=2.2, alpha=0.5, color='0.7')
ax.set_axisbelow(True)
ax.plot(forced_positions, forced_loss, color=models[profiled]['color'], linewidth=3.5,
        zorder=4, label=f"Loss with '{profiled}' forced to break here")
ax.axhline(ladder[reference]['loss'], color='0.35', linestyle='--', linewidth=2.5,
           zorder=3, label=f'{reference}, no breakpoint')
ax.axvline(ladder[profiled]['params'][break_index], color='crimson', linewidth=3,
           linestyle=(0, (6, 3)), zorder=5, label='Where the free fit landed')
ax.set_xlabel('Breakpoint position', size=12)
ax.set_ylabel('Check loss', size=12)
ax.set_title('Why the search needs several starting points', size=14)
ax.legend(loc='upper right', fontsize=10)

for ax in axes:
    ax.tick_params(axis='both', which='major', width=3, length=10, labelsize=12)
    for spine in ax.spines.values():
        spine.set_linewidth(2)

fig.suptitle('Step 2: fitting the weighted median with no bins', size=16)
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

fig.savefig('step2_fit_and_loss_profile.png', dpi=300, bbox_inches='tight')

## Step 3 — the D statistic

Compare the best structured fit against the best unstructured one:

$$D = 2N \ln \frac{Q_{\rm structured}}{Q_{\rm unstructured}}$$

**Negative D means structure fits better.** The more negative, the more the structured
family is buying.

### Where the formula comes from

Treat the check loss as coming from an asymmetric Laplace likelihood — the working
likelihood whose maximiser is the quantile regression estimator. Profiling out its scale
gives $\hat\sigma = Q/N$, and substituting back leaves
$-2\log L = 2N \ln Q + \text{const}$. The constant is the same for every model fitted to
the same data, so it cancels in the difference and $D$ is what survives. The factor of 2
is the usual deviance convention (Wilks 1938), kept so the number sits on the scale
people are used to reading.

### Why you cannot look D up in a chi-squared table

Three independent reasons, any one of which would be enough:

1. **The working likelihood is not a real one.** The asymmetric Laplace is a device for
   getting the right estimator, not a claim about how your data are distributed. The
   figure below shows how far off it is for this sample.
2. **The completeness weights break the information matrix equality.** Weighted
   estimating equations have a sandwich covariance, not the inverse-Hessian form that
   Wilks' theorem assumes.
3. **A breakpoint is unidentified under the null.** When the truth is flat, the
   breakpoint has nothing to estimate, so the regularity conditions fail outright
   (Davies 1977, 1987).

Measured consequence in the paper: a nominal 5 per cent test rejected **21.6 per cent**
of simulated nulls, and where the simulation gave $p = 0.007$ a $\chi^2_2$ table would
have claimed $p = 8 \times 10^{-7}$ — four orders of magnitude too confident. The last
panel of the final figure shows that failure directly.

![](step3_working_likelihood.png)

In [ ]:
#### THE D STATISTIC ####


def d_statistic(fitted):
    # Twice the log likelihood improvement, no model selection penalty. Negative means
    # the structured family fits better.
    best_simple = min(fitted[name]['loss'] for name in simple_models)
    best_structured = min(fitted[name]['loss'] for name in structured_models)
    return 2 * n_objects * np.log(best_structured / best_simple)


observed_d = d_statistic(ladder)

print(f"Observed D = {observed_d:+.2f}")
print(f"  from N = {n_objects}, "
      f"Q_structured = {min(ladder[n]['loss'] for n in structured_models):.3f}, "
      f"Q_simple = {min(ladder[n]['loss'] for n in simple_models):.3f}")
print("\n  Negative D means structure fits better, but how much better is")
print("  surprising cannot be read off a table -- that needs the simulation below.")

In [ ]:
#### IS THE WORKING LIKELIHOOD A REAL ONE? ####
# D is derived from an asymmetric Laplace likelihood. At tau = 0.5 its density is
# (1/4 sigma) exp(-|u| / 2 sigma) with sigma = Q/N, so it can be drawn straight onto the
# residuals it claims to describe. Where they part company is reason 1 for not trusting
# a chi-squared table, and the log panel shows it is the tails that go wrong.

best_name = min(models, key=lambda name: ladder[name]['loss'])
residuals = y - models[best_name]['func'](x, *ladder[best_name]['params'])
sigma = ladder[best_name]['loss'] / n_objects
u_line = np.linspace(residuals.min(), residuals.max(), 400)
laplace = (0.25 / sigma) * np.exp(-np.abs(u_line) / (2 * sigma))

print(f"Residuals about the best fit ({best_name})")
print(f"  sigma = Q/N = {sigma:.4f}, so the implied Laplace scale is {2*sigma:.4f}")
print(f"  weighted residual spread: 16-84 per cent "
      f"{np.percentile(residuals, 16):+.3f} to {np.percentile(residuals, 84):+.3f}")
print(f"  the Laplace predicts     16-84 per cent "
      f"{-2*sigma*np.log(1/0.32):+.3f} to {2*sigma*np.log(1/0.32):+.3f}")

# Create the plot
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

edges = np.linspace(residuals.min(), residuals.max(), 44)
for ax, log_scale in zip(axes, (False, True)):
    ax.grid(True, which='major', linewidth=2.2, alpha=0.5, color='0.7')
    ax.set_axisbelow(True)
    ax.hist(residuals, bins=edges, weights=weights, density=True, color='0.6',
            alpha=0.7, label='Weighted residuals')
    ax.plot(u_line, laplace, color='crimson', linewidth=3.5, zorder=5,
            label='Asymmetric Laplace at tau = 0.5')
    ax.set_xlabel('Residual about the fitted curve', size=12)
    ax.set_ylabel('Density', size=12)
    if log_scale:
        ax.set_yscale('log')
        ax.set_title('The same thing on a log scale: the tails', size=14)
    else:
        ax.set_title('The likelihood D is derived from', size=14)
    ax.legend(loc='upper right', fontsize=10)
    ax.tick_params(axis='both', which='major', width=3, length=10, labelsize=12)
    for spine in ax.spines.values():
        spine.set_linewidth(2)

fig.suptitle('Step 3: a working likelihood, not a description of the data', size=16)
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

fig.savefig('step3_working_likelihood.png', dpi=300, bbox_inches='tight')

## Step 4 — forward modelling the survey

This is the step that makes the test valid, and the one most easily got wrong.

### The wrong ways to do it

The obvious approach is to pick a truth curve, resample residuals onto it, refit, and
repeat. There are two ways to handle the weights and **both are wrong**:

- **Freeze each object's weight.** An object moved from 2.1 to 2.9 keeps the large
  correction it earned at 2.1, so it is corrected for a size it no longer has.
- **Recompute each weight at the new value.** This fixes that but breaks something
  worse. A fake object placed at 1.85 is handed the enormous correction that only a
  *genuinely detected* 1.85 object has earned — when nothing in the simulation ever
  asked whether a survey could have found it there.

In the paper these two schemes disagreed badly and neither was trustworthy.

### The right way

Make detection part of the generation. A real detected object is not drawn from the
intrinsic population; it is drawn from the intrinsic population **filtered by whether
the survey could see it**, and its weight is the reciprocal of that filter. Doing the
same in the simulation makes values and weights consistent by construction.

For one slot $i$, holding its $x_i$ and every other covariate at the real values:

$$
\begin{aligned}
\text{intrinsic distribution} &: g(y - \text{truth}(x_i)) \\
\text{chance of detection} &: d(i, y) \\
\text{detected distribution} &: \propto g(y - \text{truth}(x_i)) \cdot d(i, y)
\end{aligned}
$$

And $g$ is recovered from the real residuals by undoing the same filter: the *observed*
residuals follow $g(u)\,d(i, \text{truth}+u)$, so $g$ is the residual distribution
weighted by each object's own completeness weight — which is already $1/d$ at its own
position.

So the whole forward model collapses to one weighted resample. Draw residual $j$ for
slot $i$ with probability proportional to

$$w_j \times d\!\left(i,\ \text{truth}(x_i) + u_j\right)$$

the first factor **undoing** the detection that shaped the observed residual, the second
**redoing** the detection appropriate to where it is landing. Where two slots have equal
detectability the factors cancel and this reduces to plain resampling, so the whole
difference is driven by objects moving between regions of different completeness. The
first panel below draws both factors for a single slot.

Two things fall out for free: a value landing outside your selection window has $d = 0$
and is simply never drawn, and the simulated weights are then the honest ones, because
the values were drawn with that same detection probability applied.

![](step4_forward_model.png)

In [ ]:
#### FORWARD MODELLING THE SURVEY ####
# Draw residual j for slot i with probability proportional to w_j * d(i, truth_i + u_j):
# the first factor undoing the detection that shaped the observed residual, the second
# applying the detection appropriate to where it is landing.


def build_draw_probabilities(truth_curve, residual_pool):
    # For every slot i and every candidate residual j, the chance slot i ends up holding
    # a detected object built from residual j. Returned as a cumulative sum per row so a
    # survey costs one uniform draw per slot.
    candidates = truth_curve[:, None] + residual_pool[None, :]
    slot_grid = np.broadcast_to(all_slots[:, None], candidates.shape)
    detection = detectability(slot_grid.ravel(), candidates.ravel())
    probability = detection.reshape(candidates.shape) * raw_weights[None, :]
    total = probability.sum(axis=1, keepdims=True)
    # A slot that can reach nothing at all keeps a plain resample rather than vanishing
    dead = total[:, 0] == 0
    probability[dead] = 1.0
    total[dead] = len(residual_pool)
    return np.cumsum(probability / total, axis=1)


def forward_survey(cumulative, truth_curve, residual_pool, generator):
    # One simulated survey: its values, and the weights those values honestly earn
    u = generator.random(n_objects)
    picked = (cumulative < u[:, None]).sum(axis=1).clip(0, len(residual_pool) - 1)
    fake_y = truth_curve + residual_pool[picked]
    fake_weights = completeness_weight(all_slots, fake_y)
    fake_weights = np.where(np.isfinite(fake_weights), fake_weights, 1.0)
    # Divided by the constant from the real sample, not by this survey's own mean, so
    # that a loss means the same thing in every survey
    return fake_y, fake_weights / raw_mean


# Check the machinery against the real sample before trusting it. Generating from the
# best-fitting truth should reproduce the observed distribution, and the offset between
# the unweighted residuals and the fitted curve should come out on its own rather than
# being put in by hand.
check_truth = models[best_name]['func'](x, *ladder[best_name]['params'])
check_pool = y - check_truth
check_cumulative = build_draw_probabilities(check_truth, check_pool)

check_generator = np.random.default_rng(11)
sample, one_survey = [], None
for i in range(20):
    fake_y, fake_weights = forward_survey(check_cumulative, check_truth, check_pool,
                                          check_generator)
    sample.append(fake_y)
    if one_survey is None:
        one_survey = fake_y
sample = np.concatenate(sample)

print("Validation of the forward model")
print("=" * 70)
print(f"  real objects      median {np.median(y):.3f}, "
      f"16-84 per cent {np.percentile(y, 16):.2f} to {np.percentile(y, 84):.2f}")
print(f"  forward modelled  median {np.median(sample):.3f}, "
      f"16-84 per cent {np.percentile(sample, 16):.2f} to {np.percentile(sample, 84):.2f}")
print(f"\n  The real residuals sit {np.median(check_pool):+.3f} above the fitted curve, "
      f"unweighted.")
print(f"  Detection reproduces an offset of "
      f"{np.median(sample - np.tile(check_truth, 20)):+.3f} without it being put in.")

In [ ]:
#### WHAT THE FORWARD MODEL ACTUALLY DOES ####
# Panel 1 takes one slot and draws the two competing factors. Panel 2 is the validation
# above, as a picture. Panel 3 is a single simulated survey next to the real one.

# The slot with the most extreme weight, where undo and redo pull hardest against
# each other and the effect is visible
slot = int(np.argmax(weights))
candidate_y = check_truth[slot] + check_pool
order = np.argsort(candidate_y)
candidate_y = candidate_y[order]
donor_factor = raw_weights[order]
redo_factor = detectability(np.full(len(candidate_y), slot), candidate_y)
product = donor_factor * redo_factor

print(f"Slot {slot}: x = {x[slot]:.0f}, real y = {y[slot]:.3f}, weight {weights[slot]:.1f}")
print(f"  candidate values it could take: {candidate_y.min():.2f} to {candidate_y.max():.2f}")
print(f"  of {len(candidate_y)} candidates, {int(np.sum(redo_factor == 0))} are undetectable "
      f"and can never be drawn")

# Create the plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

ax = axes[0]
ax.grid(True, which='major', linewidth=2.2, alpha=0.5, color='0.7')
ax.set_axisbelow(True)
# Each factor normalised to its own maximum, since only the shapes matter here
ax.scatter(candidate_y, donor_factor / donor_factor.max(), s=16, color='steelblue',
           alpha=0.55, zorder=3, label='Undo: donor weight w_j')
ax.scatter(candidate_y, redo_factor / redo_factor.max(), s=16, color='seagreen',
           alpha=0.55, zorder=3, label='Redo: detectability at the new value')
ax.scatter(candidate_y, product / product.max(), s=26, color='crimson', alpha=0.85,
           zorder=5, label='Product: the draw probability')
ax.set_xlabel('Candidate value for this slot', size=12)
ax.set_ylabel('Relative size (each scaled to its own maximum)', size=12)
ax.set_title(f'Undo then redo, for slot {slot}', size=14)
ax.legend(loc='upper center', fontsize=9)

ax = axes[1]
ax.grid(True, which='major', linewidth=2.2, alpha=0.5, color='0.7')
ax.set_axisbelow(True)
edges_y = np.linspace(min(y.min(), sample.min()), max(y.max(), sample.max()), 40)
ax.hist(y, bins=edges_y, density=True, color='0.6', alpha=0.65,
        label=f'Real objects (N={n_objects})')
ax.hist(sample, bins=edges_y, density=True, histtype='step', linewidth=3.2,
        color='crimson', label='Forward modelled (20 surveys)')
ax.set_xlabel('y', size=12)
ax.set_ylabel('Density', size=12)
ax.set_title('Does detection reproduce the real distribution?', size=14)
ax.legend(loc='upper right', fontsize=10)

ax = axes[2]
ax.grid(True, which='major', linewidth=2.2, alpha=0.5, color='0.7')
ax.set_axisbelow(True)
ax.scatter(x, y, s=13, color='0.6', alpha=0.35, zorder=2, label='Real objects')
ax.scatter(x, one_survey, s=13, color='crimson', alpha=0.35, zorder=3,
           label='One simulated survey')
x_line = np.linspace(x_min, x_max, 600)
ax.plot(x_line, models[best_name]['func'](x_line, *ladder[best_name]['params']),
        color='black', linewidth=4, zorder=6, label=f'Generating truth ({best_name})')
ax.set_xlabel('x', size=12)
ax.set_ylabel('y', size=12)
ax.set_title('One simulated survey, object by object', size=14)
ax.legend(loc='upper left', fontsize=9)

for ax in axes:
    ax.tick_params(axis='both', which='major', width=3, length=10, labelsize=12)
    for spine in ax.spines.values():
        spine.set_linewidth(2)

fig.suptitle('Step 4: detection simulated, not corrected for afterwards', size=16)
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

fig.savefig('step4_forward_model.png', dpi=300, bbox_inches='tight')

## Step 5 — calibrating the null distribution

Now run the test. For each candidate truth:

1. Build its truth curve and residual pool from the real data.
2. Forward model `n_simulations` surveys from it.
3. Fit the full ladder to each and record D.

The **unstructured** truths give the null distribution: the p-value is the fraction of
their simulated surveys reaching a D at least as negative as yours. The **structured**
truths give the power: the fraction of surveys from a universe that genuinely has
structure in which this procedure would detect it.

Both halves matter. A test that rejects the null tells you little if you cannot also say
it would have found real structure had it been there — and in the paper the power was
only about 40 per cent, which is worth stating plainly rather than hiding.

Every truth is fitted to your own data first, so the residual pool differs between
truths. That is deliberate: each generating universe should reproduce your scatter about
*its own* curve, not somebody else's.

![](step5_null_calibration.png)

In [ ]:
#### CALIBRATING THE NULL BY SIMULATION ####
# The unstructured truths give the p-value, the structured ones the power. Every truth is
# fitted to the real data first so each reproduces the scatter about its own curve.

# The structured truths are run first and all four share one generator, so each truth
# starts where the previous one left off. Seeding inside the loop instead would hand
# every truth the same uniform draws, which correlates the four null distributions.
truth_order = structured_models + simple_models

truths = {}
for name in truth_order:
    truths[name] = {'func': models[name]['func'], 'params': ladder[name]['params'],
                    'family': models[name]['family'], 'color': models[name]['color']}

generator = np.random.default_rng(11)
simulated_d = {}
for name in truths:
    truth_curve = truths[name]['func'](x, *truths[name]['params'])
    residual_pool = y - truth_curve
    cumulative = build_draw_probabilities(truth_curve, residual_pool)
    values = []
    for _ in range(n_simulations):
        fake_y, fake_weights = forward_survey(cumulative, truth_curve, residual_pool,
                                              generator)
        values.append(d_statistic(fit_ladder(x, fake_y, fake_weights)))
    values = np.array(values)
    simulated_d[name] = values[np.isfinite(values)]
    print(f"  {name} done")

print(f"\nSurveys reaching the observed D of {observed_d:+.2f}")
print("=" * 78)
print(f"{'generating truth':<20}{'family':>13}{'median D':>11}{'reached':>12}{'p':>9}")
print("-" * 78)
for name in truths:
    values = simulated_d[name]
    reached = int(np.sum(values <= observed_d))
    print(f"{name:<20}{truths[name]['family']:>13}{np.median(values):>+11.2f}"
          f"{f'{reached}/{len(values)}':>12}{reached / len(values):>9.4f}")

# The 5 per cent critical value is the most lenient of the unstructured nulls, so the
# test is calibrated against the hardest null to reject rather than a convenient one
threshold = min(np.percentile(simulated_d[name], 5)
                for name in truths if truths[name]['family'] == 'simple')
print(f"\n5 per cent critical value from the unstructured nulls: D = {threshold:+.2f}")

print(f"\nPower: how often real structure would be detected")
print("-" * 78)
for name in truths:
    if truths[name]['family'] != 'structured':
        continue
    power = np.mean(simulated_d[name] <= threshold)
    print(f"  truth = {name:<18}{power:>7.1%} detected, {1 - power:>6.1%} missed")

# What a chi-squared table would have claimed, for the same simulated nulls. The
# likelihood ratio statistic is -D, and the naive reading compares it against chi2 with
# the difference in parameter count.
from scipy.stats import chi2

degrees = (max(models[n]['k'] for n in structured_models)
           - max(models[n]['k'] for n in simple_models))
naive_critical = chi2.ppf(0.95, degrees)
worst_null = max(truths, key=lambda n: np.mean(-simulated_d[n] > naive_critical)
                 if truths[n]['family'] == 'simple' else -1)
over_rejection = np.mean(-simulated_d[worst_null] > naive_critical)
print(f"\nWhat a chi-squared table would have claimed")
print("-" * 78)
print(f"  chi2 with {degrees} degrees of freedom, 5 per cent point: {naive_critical:.2f}")
print(f"  simulated nulls exceeding it ({worst_null}): {over_rejection:.1%}")
print(f"  so a nominal 5 per cent test actually rejects {over_rejection:.1%} of the time")
print(f"  and the observed D of {observed_d:+.2f} would be quoted as "
      f"p = {chi2.sf(-observed_d, degrees):.2g}")
simulated_p = np.mean(simulated_d[worst_null] <= observed_d)
if simulated_p == 0:
    print(f"  against the simulated p < {1 / len(simulated_d[worst_null]):.4f} "
          f"(no simulated null got there at all)")
else:
    print(f"  against the simulated p = {simulated_p:.4f}")

In [ ]:
#### THE RESULT ####

# Create the plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

ax = axes[0]
ax.grid(True, which='major', linewidth=2.2, alpha=0.5, color='0.7')
ax.set_axisbelow(True)
# Individual objects as grey transparent dots
ax.scatter(x, y, s=13, color='0.6', alpha=0.3, zorder=2, label=f'Objects (N={n_objects})')
x_line = np.linspace(x_min, x_max, 600)
for name in models:
    ax.plot(x_line, models[name]['func'](x_line, *ladder[name]['params']),
            color=models[name]['color'], linewidth=2.8, zorder=3, label=name)
ax.set_xlabel('x', size=12)
ax.set_ylabel('y', size=12)
ax.set_title('The fitted ladder', size=14)
ax.legend(loc='upper left', fontsize=10)

ax = axes[1]
ax.grid(True, which='major', linewidth=2.2, alpha=0.5, color='0.7')
ax.set_axisbelow(True)
everything = np.concatenate([simulated_d[name] for name in truths])
low, high = np.floor(np.percentile(everything, 0.5)), np.ceil(np.percentile(everything, 99.5))
edges = np.linspace(low, high, 44)
for name in truths:
    values = simulated_d[name]
    if truths[name]['family'] == 'simple':
        ax.hist(np.clip(values, low, high), bins=edges, alpha=0.5,
                color=truths[name]['color'],
                weights=np.full(len(values), 1 / len(values)),
                label=f'Unstructured truth ({name})')
    else:
        ax.hist(np.clip(values, low, high), bins=edges, histtype='step', linewidth=2.8,
                color=truths[name]['color'],
                weights=np.full(len(values), 1 / len(values)),
                label=f'Structured truth ({name})')
ax.axvline(observed_d, color='black', linewidth=4, zorder=6,
           label=f'Observed = {observed_d:.0f}')
ax.axvline(threshold, color='0.3', linestyle=':', linewidth=2.5, zorder=5,
           label=f'5 per cent point = {threshold:.0f}')
ax.set_xlabel('D statistic', size=12)
ax.set_ylabel('Fraction of simulated surveys', size=12)
ax.set_xlim(low, high)
ax.set_title('Simulated nulls against the observation', size=14)
ax.legend(loc='upper left', fontsize=8, framealpha=0.95)

ax = axes[2]
ax.grid(True, which='major', linewidth=2.2, alpha=0.5, color='0.7')
ax.set_axisbelow(True)
# The same nulls read as a likelihood ratio, against the chi-squared curve that does not
# apply to them
positive = -simulated_d[worst_null]
cut = np.percentile(positive, 99.5)
edges_chi = np.linspace(0, max(cut, naive_critical * 1.3), 40)
ax.hist(np.clip(positive, 0, edges_chi[-1]), bins=edges_chi, density=True, color='0.6',
        alpha=0.7, label=f'Simulated null (-D, truth = {worst_null})')
chi_line = np.linspace(0.05, edges_chi[-1], 300)
ax.plot(chi_line, chi2.pdf(chi_line, degrees), color='crimson', linewidth=3.5, zorder=5,
        label=f'chi2 with {degrees} degrees of freedom')
ax.axvline(naive_critical, color='crimson', linestyle='--', linewidth=3, zorder=4,
           label=f'Its 5 per cent point = {naive_critical:.1f}')
ax.axvline(np.percentile(positive, 95), color='black', linewidth=3.5, zorder=6,
           label=f'True 5 per cent point = {np.percentile(positive, 95):.1f}')
ax.set_xlabel('Likelihood ratio statistic (-D)', size=12)
ax.set_ylabel('Density', size=12)
ax.set_title(f'A nominal 5% test rejects {over_rejection:.0%} of nulls', size=14)
ax.legend(loc='upper right', fontsize=8, framealpha=0.95)

for ax in axes:
    ax.tick_params(axis='both', which='major', width=3, length=10, labelsize=12)
    for spine in ax.spines.values():
        spine.set_linewidth(2)

fig.suptitle('Step 5: is the structure more than selection and noise?', size=16)
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

fig.savefig('step5_null_calibration.png', dpi=300, bbox_inches='tight')

## Adapting this to your own sample

**Write `completeness_weight` as a lookup, indexed by slot.** For a real survey it will
read an injection-recovery grid. The pattern, from the notebook this came from, where the
grid is keyed by (temperature slice, radius cell, period cell):

```python
def completeness_weight(slots, y_values):
    cells = cell_index(np.asarray(y_values, dtype=float), radius_edges)
    found = np.full(len(cells), np.nan)
    inside = np.isfinite(cells)
    found[inside] = weight_table[np.asarray(slots)[inside], cells[inside].astype(int)]
    # Outside the grid means the survey could not have found it there at all
    return np.where(np.isfinite(found), found, np.inf)


raw_weights = planets['weight'].values.astype(float)
```

`weight_table[i, cell]` is precomputed once per object, so the temperature slice and
period cell are already baked in and only the radius cell moves. Returning infinity
outside the grid is what stops fake objects leaking out of your selection window: the
forward model takes the reciprocal, and `1/inf` is zero probability.

**Seed every generator.** `np.random.default_rng(11)` appears in every simulation cell
above. Unseeded resampling once made two cells fitting identical data appear to
disagree — the medians matched and only the error bars moved, which took a while to
diagnose.

**Give each new model several starting points.** The second panel of the Step 2 figure
is the reason: the loss surface has more than one basin in a breakpoint, and a single
start lands in whichever one it began in. The next cell's section on adding models says
how to write them.

**Watch for breakpoints collapsing onto the edge of your range.** If a fitted breakpoint
sits at `x_min` or `x_max`, the model has no plateau left and has quietly become a
simpler shape. That is informative — the data are refusing that shape — but it is not a
fit, and reporting its parameters as though it were would mislead. The loss profile in
Step 2 is how you check.

**Check your effective sample size before trusting a region.** A region whose `n_eff` is
a small fraction of its object count is being carried by a few heavily weighted objects.

**D compares families, not members within a family.** It will tell you that structure
beats no structure. It will *not* reliably tell you which structured shape is right, or
where the structure sits — two different structured truths can produce similarly extreme
D at similar rates. For that question use the loss directly, or look at where the
unstructured model fails region by region.

**Report the power alongside the p-value.** A p-value says the null is unlikely. The
power says whether your survey could have found the effect at all. Both belong in the
paper.

## Adding your own models

Everything downstream of the ladder reads the `models` dict, so a new shape is one entry
and no other edits. A new model is fitted to the real sample, becomes a member of its
family for the `D` statistic, and **automatically becomes a generating truth in Step 5** —
the cost is one more block of `n_simulations` surveys.

An entry needs six things:

| Key | What it is |
|---|---|
| `func` | `func(x_values, *params)`, vectorised over `x_values` |
| `k` | how many parameters, used only for the chi-squared comparison |
| `family` | `'simple'` or `'structured'` |
| `starts` | `starts(sample) -> list of parameter vectors` |
| `breaks` | which parameter positions are **positions on the x axis** |
| `color` | its colour in every figure |

**`breaks` is the only part that carries a constraint.** Any position listed there is
clipped into the data range, and where more than one is listed they must come out in
increasing order or the start is discarded. That rule is written for a list of any
length, so a three-segment model needs `'breaks': [1, 2, 3]` and nothing else.

**`starts` takes a summary of the sample being fitted,** not the real one. `sample` holds
`x`, `y`, `weights`, `level` (the median of `y`), `gradient` (the difference in median `y`
between the upper and lower halves of `x`, per unit `x`), `x_min`, `x_max` and `span`.
Build the starting points from those rather than from numbers you read off your own plot,
or the simulated surveys will be started from the wrong place — Step 5 fits thousands of
samples with this same function, and each has its own level and gradient.

### Two worked examples

A break at the **hot** end only, the mirror of `one break`. Worth having whenever you
cannot assume in advance which end of the range bends:

```python
def hot_break(x_values, c, x2, s2):
    # Flat at c below x2, sloping above it
    x_values = np.asarray(x_values, dtype=float)
    return np.where(x_values > x2, c + s2 * (x_values - x2), c)


def hot_break_starts(sample):
    starts = []
    for fraction in (0.2, 0.35, 0.5, 0.65, 0.8):
        starts.append([sample['level'], x_min + fraction * span, sample['gradient']])
    return starts


add_model('hot break', func=hot_break, k=3, family='structured',
          starts=hot_break_starts, breaks=[1], color='teal')
```

Two sloping sections with **independent** gradients, which drops the assumption that the
two ends bend by the same amount:

```python
def asymmetric(x_values, c, x1, x2, s1, s2):
    # Slope s1 below x1, flat at c between the breaks, slope s2 above x2
    x_values = np.asarray(x_values, dtype=float)
    return np.select(
        [x_values < x1, (x_values >= x1) & (x_values <= x2), x_values > x2],
        [c + s1 * (x_values - x1), np.full_like(x_values, c), c + s2 * (x_values - x2)],
    )


def asymmetric_starts(sample):
    starts = []
    for lower, upper in ((0.2, 0.8), (0.3, 0.7), (0.15, 0.55), (0.45, 0.85), (0.35, 0.6)):
        starts.append([sample['level'], x_min + lower * span, x_min + upper * span,
                       sample['gradient'], sample['gradient']])
    return starts


add_model('asymmetric', func=asymmetric, k=5, family='structured',
          starts=asymmetric_starts, breaks=[1, 2], color='rebeccapurple')
```

A model with no plateau at all — two slopes meeting at a single break — is the same
pattern with `breaks=[1]` and both gradients free.

### Where to put it

Either way works, and they differ only in what survives a re-run.

**In a new cell, with `add_model`.** Put the block in a cell directly below the ladder
cell and run everything from there down. `add_model` registers the shape, refreshes the
family lists and refits the ladder, so the rest of the notebook picks it up. Do **not**
re-run the ladder cell afterwards — it rebuilds `models` from scratch and your model
disappears. This is the route to use if you want the shipped cell left alone.

**Or as an entry in the ladder cell itself,** alongside the four already there, in the dict
form the cell uses. That survives anything, and is what to do once you have settled on
the shapes you want.

### Three things to know before you add one

**The simple family should nest inside the structured one.** $D$ is read as "what does
structure buy", and that reading depends on every simple shape being reachable as a
special case of some structured shape. A structured model that cannot reduce to a flat
line makes $D$ a comparison of two different families rather than a test for structure.

**Adding a structured model cannot manufacture a result.** It makes the observed $D$ more
negative, but it also lets every simulated null chase noise harder, so the null
distribution moves with it. In the paper's own test of this, adding a five-parameter model
moved the observed $D$ from $-28.04$ to $-31.05$ while the p-values barely shifted:

| generating truth | p, four models | p, five models |
|---|---|---|
| flat plateau | 0.0070 | 0.0055 |
| straight line | 0.0020 | 0.0050 |

That is the calibration doing its job, and it is why the simulation cannot be gamed by
enlarging the ladder. It also means an extra model costs you simulation time and buys
very little, unless it is a shape you have a physical reason to want.

**A fit that lands a breakpoint on the edge of the range is not a fit.** It has quietly
become a simpler shape, with a segment holding almost no objects and a slope free to take
any value. The ladder checks in the fitting cell print a warning when this happens; 6.5
per cent of the paper's five-parameter bootstrap resamples did it. Treat those as
optimiser failures, not as better fits.

### If you are working on a different sample

Nothing in the generic cells hardcodes a radius range or a temperature range. The
selection window is wherever your completeness grid has values — `completeness_weight`
returns `inf` outside it, the forward model takes the reciprocal, and a simulated object
can never land there. Change the grid and the window changes with it.

Two things to watch on a smaller or narrower sample. The per-region `n_eff` table in Step
1 matters more, because a handful of heavily weighted objects can carry a whole stretch of
the range. And the power in Step 5 will be lower, which is a fact about the sample rather
than a problem with the test — report it either way.

## Reproducing a published run

Plugging a sample into the block in the first cell reproduces that sample's published
numbers survey for survey, not just distribution for distribution. Checked against the
cached run behind the paper this came from: the same 673 planets give the same observed
`D = -28.0404332136`, the same four fitted shapes to every printed digit, and all 4000
simulated `D` values bit-identical.

Five things have to hold for that, and each was found by having it not hold.

**The weights have to be built by the same arithmetic.** `1/detection_rate * 1/transit_prob`
and `1/(detection_rate * transit_prob)` differ in the last bit for about one object in
seven. That was enough to move the fitted breakpoints by 14 K and 7 K, scrambling the
truths and with them every simulated survey. The loss is piecewise linear and nearly flat
near its minimum, so Powell stops wherever a 1e-16 nudge sends it. This is also worth
knowing on its own: it puts a floor of roughly +/-15 K on a breakpoint, before any
statistical uncertainty, so quoting one to the degree claims more than the optimiser can
deliver.

**The starting points have to come from the sample being fitted,** not from the real one.
`fit_model` recomputes `level` and `gradient` from its arguments on every call. Freezing
them at the real sample's values changed individual simulated `D` by as much as 35,
because the search started in a different basin.

**All the truths have to share one generator,** so each starts where the last one
finished. Re-seeding inside the loop hands every truth the same uniform draws, which
correlates the four null distributions.

**The truths have to run in the same order.** `truth_order = structured_models +
simple_models` is what the published cell does, and the order fixes which part of the
stream each truth gets.

**`n_simulations` has to match.** Not only for the count: because the generator is shared,
a shorter run leaves every truth after the first starting from a different point. A run
of 200 reproduces the first truth exactly and none of the others.

One thing that does *not* have to match: the simulated weights are divided by the
constant `raw_mean` from the real sample rather than by each survey's own mean. A global
rescale cancels out of `D` exactly, so either choice gives the same answer -- but it
leaves losses comparable between surveys, which is worth having.

SciPy's Powell implementation is the one piece of this that is outside the notebook. A
change to it would move the fits, so pin the version alongside the seed if the exact
numbers matter.

## References

- Koenker & Bassett 1978, *Econometrica* **46**, 33 — quantile regression
- Wilks 1938, *Ann. Math. Stat.* **9**, 60 — the likelihood ratio statistic and the factor of 2
- Davies 1977, *Biometrika* **64**, 247; 1987, *Biometrika* **74**, 33 — a nuisance parameter unidentified under the null
- Protassov et al. 2002, *ApJ* **571**, 545 — calibrating a likelihood ratio by simulation in an astronomical setting
- Kish 1965, *Survey Sampling* — effective sample size under weighting